# Практична робота №4
**Тема:** Агрегація та групування

**Виконав:** Онукевич Данило, КН-41

**Набір даних:** продажі кави в торговому автоматі (`index_1.csv`).

Ноутбук має три частини: кроки практичної частини (1-10), індивідуальне завдання за варіантом (№2) та додаткове завдання ★.

# Частина 1. Практична частина

## Крок 1. Підготовка середовища та завантаження даних

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/index_1.csv')
df.head()

Mounted at /content/drive


,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


## Крок 2. Групування за однією колонкою та агрегація
Метод `groupby()` лише розділяє дані на групи, а обчислення виконує агрегатна функція (`sum()`, `mean()` тощо). Результат — `Series`, де індекс — назви напоїв.

In [2]:
revenue_by_coffee = df.groupby('coffee_name')['money'].sum()
print(revenue_by_coffee)

coffee_name
Americano              15062.26
Americano with Milk    25269.12
Cappuccino             18034.14
Cocoa                   8678.16
Cortado                 7534.86
Espresso                2814.28
Hot Chocolate          10172.46
Latte                  27866.30
Name: money, dtype: float64


In [3]:
avg_money = df.groupby('coffee_name')['money'].mean()
print(avg_money)

coffee_name
Americano              26.059273
Americano with Milk    30.666408
Cappuccino             35.996287
Cocoa                  35.712593
Cortado                25.804315
Espresso               21.002090
Hot Chocolate          36.072553
Latte                  35.634655
Name: money, dtype: float64


## Крок 3. Групування за кількома колонками
Групуємо за датою та напоєм; `size()` повертає кількість рядків у кожній групі. Результат має багаторівневий індекс, який `reset_index()` перетворює на звичайну таблицю.

In [4]:
trans_count = df.groupby(['date', 'coffee_name']).size()
print(trans_count)

date        coffee_name        
2024-03-01  Americano              1
            Americano with Milk    4
            Cocoa                  1
            Hot Chocolate          3
            Latte                  2
                                  ..
2025-03-22  Latte                  1
2025-03-23  Americano              1
            Cappuccino             1
            Cocoa                  2
            Latte                  2
Length: 1711, dtype: int64


In [5]:
trans_count_df = df.groupby(['date', 'coffee_name']).size().reset_index(name='count')
print(trans_count_df.head(10))

         date          coffee_name  count
0  2024-03-01            Americano      1
1  2024-03-01  Americano with Milk      4
2  2024-03-01                Cocoa      1
3  2024-03-01        Hot Chocolate      3
4  2024-03-01                Latte      2
5  2024-03-02            Americano      3
6  2024-03-02  Americano with Milk      3
7  2024-03-02                Latte      1
8  2024-03-03            Americano      1
9  2024-03-03  Americano with Milk      2


## Крок 4. Кілька агрегатних функцій через `agg()`

In [6]:
summary = df.groupby('coffee_name')['money'].agg(['sum', 'mean', 'min', 'max'])
print(summary)

                          sum       mean    min   max
coffee_name                                          
Americano            15062.26  26.059273  23.02  30.0
Americano with Milk  25269.12  30.666408  27.92  35.0
Cappuccino           18034.14  35.996287  32.82  40.0
Cocoa                 8678.16  35.712593  32.82  40.0
Cortado               7534.86  25.804315  23.02  30.0
Espresso              2814.28  21.002090  18.12  25.0
Hot Chocolate        10172.46  36.072553  32.82  40.0
Latte                27866.30  35.634655  32.82  40.0


## Крок 5. Зведена таблиця (`pivot_table`)
Додаємо день тижня (понеділок = 0, неділя = 6) і будуємо таблицю: рядки — напої, стовпці — дні тижня, значення — сума продажів.

In [7]:
df['day_of_week'] = pd.to_datetime(df['datetime']).dt.dayofweek

pivot = pd.pivot_table(df, values='money', index='coffee_name',
                       columns='day_of_week', aggfunc='sum', fill_value=0)
print(pivot)

day_of_week                0        1        2        3        4        5  \
coffee_name                                                                 
Americano            2568.12  2168.62  2309.56  2153.82  2766.58  1847.28   
Americano with Milk  4053.78  4436.60  3479.06  3156.04  3230.90  3754.86   
Cappuccino           2676.58  2375.80  2711.72  2827.62  2343.00  2571.92   
Cocoa                1205.06  2074.38   929.76   819.54  1905.40   894.60   
Cortado               948.40  1162.44   834.76  1069.74   966.40  1418.48   
Espresso              259.62   327.16   587.28   563.72   440.30   270.34   
Hot Chocolate        1246.70  1805.96  1307.28  1766.26  1612.14   989.02   
Latte                4966.84  4286.42  3934.04  4120.66  3992.94  3436.02   

day_of_week                6  
coffee_name                   
Americano            1248.28  
Americano with Milk  3157.88  
Cappuccino           2527.50  
Cocoa                 849.42  
Cortado              1134.64  
Espresso    

## Крок 6. Перехресна таблиця (`crosstab`)
Підраховує частоту комбінацій двох категоріальних змінних: скільки транзакцій кожного напою оплачено карткою та готівкою.

In [8]:
crosstab = pd.crosstab(df['coffee_name'], df['cash_type'])
print(crosstab)

cash_type            card  cash
coffee_name                    
Americano             564    14
Americano with Milk   809    15
Cappuccino            486    15
Cocoa                 239     4
Cortado               287     5
Espresso              129     5
Hot Chocolate         276     6
Latte                 757    25


## Крок 7. Об'єднання через `merge`
Таблиця категорій містить лише чотири напої. При `how='left'` усі рядки `df` зберігаються, а для напоїв, яких немає в `categories`, у колонці `category` буде `NaN`.

In [9]:
categories = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Hot Chocolate', 'Cappuccino'],
    'category': ['молочний', 'чорний', 'молочний', 'молочний']
})

df_merged = df.merge(categories, on='coffee_name', how='left')
print(df_merged.head())
print("Рядків без категорії:", df_merged['category'].isnull().sum())

         date                 datetime cash_type                 card  money  \
0  2024-03-01  2024-03-01 10:15:50.520      card  ANON-0000-0000-0001   38.7   
1  2024-03-01  2024-03-01 12:19:22.539      card  ANON-0000-0000-0002   38.7   
2  2024-03-01  2024-03-01 12:20:18.089      card  ANON-0000-0000-0002   38.7   
3  2024-03-01  2024-03-01 13:46:33.006      card  ANON-0000-0000-0003   28.9   
4  2024-03-01  2024-03-01 13:48:14.626      card  ANON-0000-0000-0004   38.7   

     coffee_name  day_of_week  category  
0          Latte            4  молочний  
1  Hot Chocolate            4  молочний  
2  Hot Chocolate            4  молочний  
3      Americano            4    чорний  
4          Latte            4  молочний  
Рядків без категорії: 1493


## Крок 8. Об'єднання через `concat`
`concat()` ставить таблиці одну під одною. Розбиваємо датасет за датою на дві частини і склеюємо назад.

In [10]:
df_part1 = df[df['date'] <= '2024-03-15']
df_part2 = df[df['date'] > '2024-03-15']

df_concat = pd.concat([df_part1, df_part2])
print(f"Розміри: {len(df_part1)} + {len(df_part2)} = {len(df_concat)}")

Розміри: 120 + 3516 = 3636


## Крок 9. Об'єднання через `join`
`join()` з'єднує таблиці за індексами (за замовчуванням — ліве об'єднання).

In [11]:
left = pd.DataFrame({'A': [1, 2, 3]}, index=['a', 'b', 'c'])
right = pd.DataFrame({'B': [10, 20, 30]}, index=['a', 'b', 'c'])

joined = left.join(right)
print(joined)

   A   B
a  1  10
b  2  20
c  3  30


## Крок 10. Аналітичний показник: частка доходу кожного напою

In [12]:
total_revenue = df['money'].sum()
revenue_share = (df.groupby('coffee_name')['money'].sum() / total_revenue * 100).round(2)
print(revenue_share)

coffee_name
Americano              13.05
Americano with Milk    21.89
Cappuccino             15.62
Cocoa                   7.52
Cortado                 6.53
Espresso                2.44
Hot Chocolate           8.81
Latte                  24.14
Name: money, dtype: float64


**Інтерпретація:** найбільшу частку доходу дають Latte (близько 24%) та Americano with Milk (близько 22%), а найменшу — Espresso (близько 2.4%).

# Частина 2. Індивідуальне завдання (варіант 2)

## Завдання 2. Максимальна та мінімальна сума транзакції для кожного напою
Групуємо за `coffee_name` і застосовуємо `agg()` з функціями `max` та `min` до колонки `money`.

In [13]:
max_min = df.groupby('coffee_name')['money'].agg(['max', 'min'])
print(max_min)

                      max    min
coffee_name                     
Americano            30.0  23.02
Americano with Milk  35.0  27.92
Cappuccino           40.0  32.82
Cocoa                40.0  32.82
Cortado              30.0  23.02
Espresso             25.0  18.12
Hot Chocolate        40.0  32.82
Latte                40.0  32.82


Для наочності можна додати розмах (різницю між максимумом і мінімумом) та відсортувати за максимумом.

In [14]:
max_min['range'] = max_min['max'] - max_min['min']
max_min.sort_values('max', ascending=False)

,max,min,range
coffee_name,,,
Cocoa,40.0,32.82,7.18
Cappuccino,40.0,32.82,7.18
Hot Chocolate,40.0,32.82,7.18
Latte,40.0,32.82,7.18
Americano with Milk,35.0,27.92,7.08
Americano,30.0,23.02,6.98
Cortado,30.0,23.02,6.98
Espresso,25.0,18.12,6.88


**Висновок:** найдорожчі транзакції мають значення 40.0 (Latte, Cappuccino, Cocoa, Hot Chocolate), а найдешевший напій Espresso має максимум лише 25.0 і мінімум 18.12.

# Частина 3. ★ Додаткове завдання
Власний набір даних із двох таблиць: **товари** (`products`) та **замовлення** (`orders`). З'єднуємо їх через `merge()`, групуємо, рахуємо суму, середнє та кількість, будуємо зведену таблицю й зберігаємо результати у CSV.

### Крок 1. Дві таблиці

In [15]:
products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5],
    'product': ['Latte', 'Americano', 'Cappuccino', 'Cocoa', 'Espresso'],
    'category': ['молочний', 'чорний', 'молочний', 'десертний', 'чорний'],
    'price': [38.7, 28.9, 36.0, 35.7, 21.0]
})

orders = pd.DataFrame({
    'order_id': range(1, 21),
    'product_id': [1, 2, 1, 3, 4, 5, 2, 1, 3, 2, 4, 1, 5, 3, 2, 1, 4, 3, 2, 5],
    'quantity': [2, 1, 1, 3, 2, 1, 2, 1, 1, 3, 1, 2, 2, 1, 1, 3, 1, 2, 2, 1],
    'city': ['Kyiv', 'Lviv', 'Kyiv', 'Odesa', 'Lviv', 'Kyiv', 'Odesa', 'Lviv', 'Kyiv', 'Odesa',
             'Kyiv', 'Lviv', 'Odesa', 'Lviv', 'Kyiv', 'Odesa', 'Lviv', 'Kyiv', 'Lviv', 'Odesa']
})
print(products)
print(orders.head())

   product_id     product   category  price
0           1       Latte   молочний   38.7
1           2   Americano     чорний   28.9
2           3  Cappuccino   молочний   36.0
3           4       Cocoa  десертний   35.7
4           5    Espresso     чорний   21.0
   order_id  product_id  quantity   city
0         1           1         2   Kyiv
1         2           2         1   Lviv
2         3           1         1   Kyiv
3         4           3         3  Odesa
4         5           4         2   Lviv


### Крок 2. З'єднання таблиць (`merge`) та нова колонка виручки

In [16]:
data = orders.merge(products, on='product_id', how='left')
data['revenue'] = data['quantity'] * data['price']
data.head()

,order_id,product_id,quantity,city,product,category,price,revenue
0,1,1,2,Kyiv,Latte,молочний,38.7,77.4
1,2,2,1,Lviv,Americano,чорний,28.9,28.9
2,3,1,1,Kyiv,Latte,молочний,38.7,38.7
3,4,3,3,Odesa,Cappuccino,молочний,36.0,108.0
4,5,4,2,Lviv,Cocoa,десертний,35.7,71.4


### Крок 3. Групування та три агрегатні показники
Для кожної категорії рахуємо суму виручки, середню виручку замовлення та кількість замовлень.

In [17]:
category_summary = data.groupby('category').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    orders_count=('order_id', 'count'),
    items_sold=('quantity', 'sum')
).round(2)
category_summary

,total_revenue,avg_revenue,orders_count,items_sold
category,,,,
десертний,142.8,47.60,3,4
молочний,600.3,66.70,9,16
чорний,344.1,43.01,8,13


### Крок 4. Зведена таблиця
Рядки — категорії, стовпці — міста, значення — сума виручки.

In [18]:
pivot_revenue = pd.pivot_table(data, values='revenue', index='category',
                               columns='city', aggfunc='sum', fill_value=0).round(2)
pivot_revenue

city,Kyiv,Lviv,Odesa
category,,,
десертний,35.7,107.1,0.0
молочний,224.1,152.1,224.1
чорний,49.9,86.7,207.5


### Крок 5. Збереження результатів у CSV
Індекси (категорії) зберігаємо, бо вони містять важливу інформацію. Файли потім копіюємо до репозиторію, у папку `lab4/results`.

In [19]:
import os
out_dir = '/content/drive/MyDrive/lab4_results'
os.makedirs(out_dir, exist_ok=True)

data.to_csv(f'{out_dir}/orders_merged.csv', index=False)
category_summary.to_csv(f'{out_dir}/category_summary.csv')
pivot_revenue.to_csv(f'{out_dir}/pivot_revenue.csv')
print("Файли збережено в", out_dir)

Файли збережено в /content/drive/MyDrive/lab4_results


**Підсумок:** ми з'єднали дві таблиці, обчислили суму, середнє та кількість за категоріями, побудували зведену таблицю за категоріями й містами та зберегли результати.